#### Silver cleanup

 Reads the raw, messy bronze CSVs, resolves each class of data-quality issue,
 and writes clean, typed, deduplicated Delta tables to the Lakehouse Tables area.

 This is the transformation layer: the mess that was faithfully preserved in
 bronze gets resolved here, with each fix done explicitly so the logic is
 auditable.

#### Cell 1

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from functools import reduce

# calendar-safe parse + write (same settings silver needs)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

BRONZE = "Files/bronze/construction_raw"

# Read every bronze CSV as raw strings -- we do ALL typing ourselves, so Spark's
# schema inference never gets a chance to silently coerce or drop the mess.
def read_bronze(name):
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)   # everything comes in as string, on purpose
            .csv(f"{BRONZE}/{name}.csv"))

raw = {
    name: read_bronze(name)
    for name in ["projects", "subcontractors", "cost_line_items",
                 "schedule_tasks", "labor_timesheets", "sub_bids", "safety_incidents"]
}

for name, df in raw.items():
    print(f"{name:20s} {df.count():>7,} raw rows")

StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 3, Finished, Available, Finished, False)

projects                 122 raw rows
subcontractors            92 raw rows
cost_line_items        1,379 raw rows
schedule_tasks         1,440 raw rows
labor_timesheets       3,411 raw rows
sub_bids               1,917 raw rows
safety_incidents         189 raw rows


#### Cell 2

In [2]:
# Reusable cleaning helpers.
def parse_messy_date(colname):
    """
    Bronze dates arrive in many formats (yyyy-MM-dd, MM/dd/yyyy, dd-MMM-yyyy, etc.)
    plus a few impossible values. try_to_timestamp returns null on anything it
    can't parse, so impossible dates become null instead of blowing up the job.
    We coalesce across the known formats and cast to DateType.
    """
    c = F.col(colname)
    formats = ["yyyy-MM-dd", "MM/dd/yyyy", "dd-MMM-yyyy", "MM-dd-yy", "yyyy/MM/dd", "MMM dd, yyyy"]
    attempts = [F.try_to_timestamp(c, F.lit(fmt)) for fmt in formats]
    return F.coalesce(*attempts).cast(T.DateType())

def clean_str(colname):
    """Trim whitespace, collapse internal doubles, null out empties."""
    c = F.trim(F.regexp_replace(F.col(colname), r"\s+", " "))
    return F.when(c == "", None).otherwise(c)

def to_bool(colname):
    """Map the zoo of boolean encodings (Y/Yes/TRUE/N/No/FALSE) to real booleans."""
    c = F.upper(F.trim(F.col(colname).cast("string")))
    return (F.when(c.isin("Y", "YES", "TRUE", "T", "1"), F.lit(True))
             .when(c.isin("N", "NO", "FALSE", "F", "0"), F.lit(False))
             .otherwise(None))

def num(colname, cast_to="double"):
    """Cast a string column to numeric; non-numeric -> null."""
    return F.col(colname).cast(cast_to)

StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 4, Finished, Available, Finished, False)

#### Cell 3

In [3]:
# Clean PROJECTS (dates, dedupe, types, cost validity flag)
p = raw["projects"]

projects_clean = (p
    .withColumn("project_name",     clean_str("project_name"))
    .withColumn("project_type",     clean_str("project_type"))
    .withColumn("region",           clean_str("region"))
    .withColumn("delivery_method",  clean_str("delivery_method"))
    .withColumn("project_manager",  clean_str("project_manager"))
    .withColumn("contract_value",   num("contract_value"))
    .withColumn("square_footage",   num("square_footage", "int"))
    .withColumn("start_date",        parse_messy_date("start_date"))
    .withColumn("planned_end_date",  parse_messy_date("planned_end_date"))
    .withColumn("actual_end_date",   parse_messy_date("actual_end_date"))
    # data-quality flag: contract value should be positive; flag rather than drop
    .withColumn("contract_value_valid",
                F.when(F.col("contract_value") > 0, True).otherwise(False))
    # dedupe: bronze injected ~2% exact-duplicate rows
    .dropDuplicates(["project_id"])
)

print(f"projects: {p.count():,} raw -> {projects_clean.count():,} deduped")
projects_clean.select("project_id", "start_date", "planned_end_date",
                      "contract_value", "contract_value_valid").show(5, truncate=False)

StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 5, Finished, Available, Finished, False)

projects: 122 raw -> 120 deduped
+----------+----------+----------------+--------------+--------------------+
|project_id|start_date|planned_end_date|contract_value|contract_value_valid|
+----------+----------+----------------+--------------+--------------------+
|PRJ-00066 |2025-09-07|2027-04-16      |-2970981.45   |false               |
|PRJ-00061 |2022-04-08|2022-12-31      |4.001218477E7 |true                |
|PRJ-00011 |2023-01-24|2025-05-27      |4.486120249E7 |true                |
|PRJ-00018 |2022-10-16|2023-09-12      |2.514803753E7 |true                |
|PRJ-00001 |2024-06-25|2027-01-20      |1.875634659E7 |true                |
+----------+----------+----------------+--------------+--------------------+
only showing top 5 rows



#### Cell 4

In [4]:
# Clean SUBCONTRACTORS (entity resolution on dirty vendor names)
# This is the interesting one: bronze has the same vendor under variant spellings
# ("Sawyer Group LLC" / "SAWYER GROUP L.L.C." / "  sawyer group llc ") plus
# explicit -DUP rows. We build a normalized match key and collapse to one row
# per real vendor.
s = raw["subcontractors"]

subs_normalized = (s
    .withColumn("sub_name",       clean_str("sub_name"))
    .withColumn("trade_focus",    clean_str("trade_focus"))
    .withColumn("region",         clean_str("region"))
    .withColumn("performance_rating", num("performance_rating"))
    .withColumn("prequalified",   to_bool("prequalified"))
    # normalized match key: uppercase, strip punctuation, standardize legal suffixes
    .withColumn("match_key",
        F.regexp_replace(
            F.regexp_replace(
                F.upper(F.trim(F.col("sub_name"))),
                r"[.,]", ""),                       # drop periods/commas: L.L.C -> LLC
            r"\s+", " ")                             # collapse whitespace
    )
    # collapse the legal-suffix variants so LLC / L L C / INC / INC. align
    .withColumn("match_key",
        F.trim(F.regexp_replace(F.col("match_key"), r"\bL L C\b", "LLC")))
)

# Resolve to one surviving row per match_key. Prefer the row with the highest
# performance_rating (arbitrary but deterministic tie-break), keep its real sub_id.
from pyspark.sql.window import Window
w = Window.partitionBy("match_key").orderBy(F.col("performance_rating").desc_nulls_last(),
                                            F.col("sub_id").asc())

subs_clean = (subs_normalized
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print(f"subcontractors: {s.count():,} raw -> {subs_clean.count():,} resolved "
      f"({s.count() - subs_clean.count():,} duplicates collapsed)")
subs_clean.select("sub_id", "sub_name", "match_key", "prequalified").show(8, truncate=False)

StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 6, Finished, Available, Finished, False)

subcontractors: 92 raw -> 80 resolved (12 duplicates collapsed)
+--------+-----------------------------------------+----------------------------------------+------------+
|sub_id  |sub_name                                 |match_key                               |prequalified|
+--------+-----------------------------------------+----------------------------------------+------------+
|SUB-0025|Armstrong-Andrews Electric Co            |ARMSTRONG-ANDREWS ELECTRIC CO           |false       |
|SUB-0053|Avery, Horton and Fernandez Electric Co  |AVERY HORTON AND FERNANDEZ ELECTRIC CO  |NULL        |
|SUB-0055|bass plc construction                    |BASS PLC CONSTRUCTION                   |true        |
|SUB-0032|Becker, Taylor and Davis Construction    |BECKER TAYLOR AND DAVIS CONSTRUCTION    |false       |
|SUB-0028|Beltran, Lozano and Mcgee Contractors    |BELTRAN LOZANO AND MCGEE CONTRACTORS    |false       |
|SUB-0004|Booker, Jones and Harrington Construction|BOOKER JONES AND HARRINGTON 

#### Cell 5

In [5]:
# Clean COST LINE ITEMS (numeric coercion, cost validity, computed fields)
c = raw["cost_line_items"]

costs_clean = (c
    .withColumn("csi_division",        clean_str("csi_division"))
    .withColumn("division_name",       clean_str("division_name"))
    .withColumn("cost_code_notes",     clean_str("cost_code_notes"))
    .withColumn("budget_amount",       num("budget_amount"))
    .withColumn("actual_amount",       num("actual_amount"))
    .withColumn("change_order_amount", num("change_order_amount"))
    # flag invalid actuals (bronze injected null / negative / zero)
    .withColumn("actual_valid",
                F.when(F.col("actual_amount") > 0, True).otherwise(False))
    # computed analytics fields (only meaningful when both sides are valid)
    .withColumn("variance",
                F.when((F.col("budget_amount") > 0) & (F.col("actual_amount") > 0),
                       F.col("actual_amount") - F.col("budget_amount")))
    .withColumn("overrun_ratio",
                F.when((F.col("budget_amount") > 0) & (F.col("actual_amount") > 0),
                       F.round(F.col("actual_amount") / F.col("budget_amount"), 4)))
    .dropDuplicates(["line_item_id"])
)

print(f"cost_line_items: {c.count():,} raw -> {costs_clean.count():,} clean")
costs_clean.select("line_item_id", "budget_amount", "actual_amount",
                   "overrun_ratio", "actual_valid").show(5)

StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 7, Finished, Available, Finished, False)

cost_line_items: 1,379 raw -> 1,379 clean
+------------+-------------+-------------+-------------+------------+
|line_item_id|budget_amount|actual_amount|overrun_ratio|actual_valid|
+------------+-------------+-------------+-------------+------------+
| CLI-0000152|      54859.8|     52392.95|        0.955|        true|
| CLI-0000162|     66678.54|     71065.79|       1.0658|        true|
| CLI-0000555|   4035232.17|   5093682.41|       1.2623|        true|
| CLI-0000672|   1074411.34|   1299151.78|       1.2092|        true|
| CLI-0001176|   1268119.83|   1585705.48|       1.2504|        true|
+------------+-------------+-------------+-------------+------------+
only showing top 5 rows



#### Cell 6

In [6]:
# Clean SCHEDULE, LABOR, BIDS, SAFETY
sched = (raw["schedule_tasks"]
    .withColumn("task_name",              clean_str("task_name"))
    .withColumn("predecessor_task",       clean_str("predecessor_task"))
    .withColumn("planned_start",          parse_messy_date("planned_start"))
    .withColumn("planned_duration_days",  num("planned_duration_days", "int"))
    .withColumn("actual_duration_days",   num("actual_duration_days", "int"))
    .withColumn("percent_complete",       num("percent_complete", "int"))
    # schedule slip = actual - planned duration (a key delay-model feature)
    .withColumn("duration_slip_days",
                F.col("actual_duration_days") - F.col("planned_duration_days"))
    .dropDuplicates(["task_id"])
)

labor = (raw["labor_timesheets"]
    .withColumn("worker_name",     clean_str("worker_name"))
    .withColumn("trade",           clean_str("trade"))
    .withColumn("work_date",       parse_messy_date("work_date"))
    .withColumn("regular_hours",   num("regular_hours"))
    .withColumn("overtime_hours",  num("overtime_hours"))
    .withColumn("hourly_rate",     num("hourly_rate"))
    # computed labor cost (the total column bronze deliberately omitted)
    .withColumn("total_cost",
                F.round((F.coalesce(F.col("regular_hours"), F.lit(0))
                         + F.coalesce(F.col("overtime_hours"), F.lit(0)) * F.lit(1.5))
                        * F.col("hourly_rate"), 2))
    .dropDuplicates(["timesheet_id"])
)

bids = (raw["sub_bids"]
    .withColumn("csi_division", clean_str("csi_division"))
    .withColumn("bid_amount",   num("bid_amount"))
    .withColumn("awarded",      to_bool("awarded"))
    .withColumn("bid_date",     parse_messy_date("bid_date"))
    .dropDuplicates(["bid_id"])
)

safety = (raw["safety_incidents"]
    .withColumn("incident_type",  clean_str("incident_type"))
    .withColumn("severity",       clean_str("severity"))
    .withColumn("trade_involved", clean_str("trade_involved"))
    .withColumn("root_cause",     clean_str("root_cause"))
    .withColumn("description",    clean_str("description"))
    .withColumn("incident_date",  parse_messy_date("incident_date"))
    .withColumn("lost_days",      num("lost_days", "int"))
    # binary target for the safety classifier
    .withColumn("is_lost_time",
                F.when(F.col("severity") == "Lost Time", True).otherwise(False))
    .dropDuplicates(["incident_id"])
)

for nm, df in [("schedule_tasks", sched), ("labor_timesheets", labor),
               ("sub_bids", bids), ("safety_incidents", safety)]:
    print(f"{nm:20s} {df.count():>7,} clean rows")

StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 8, Finished, Available, Finished, False)

schedule_tasks         1,440 clean rows
labor_timesheets       3,411 clean rows
sub_bids               1,917 clean rows
safety_incidents         189 clean rows


#### Cell 7

In [7]:
# Stamp lineage, then write all silver tables as managed Delta
# Lineage columns make every row traceable to its batch and load time, and
# define a provenance-based train/test split: the original load is the training
# set; later incremental batches (notebook 03) arrive as test data.
silver = {
    "silver_projects":        projects_clean,
    "silver_subcontractors":  subs_clean.drop("match_key"),  # match_key was a working column
    "silver_cost_line_items": costs_clean,
    "silver_schedule_tasks":  sched,
    "silver_labor_timesheets": labor,
    "silver_sub_bids":        bids,
    "silver_safety_incidents": safety,
}

def add_lineage(df, batch_id="B1", split="train"):
    return (df
        .withColumn("batch_id",    F.lit(batch_id))
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("data_split",  F.lit(split)))

for tbl, df in silver.items():
    (add_lineage(df, batch_id="B1", split="train").write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tbl))
    print(f"wrote {tbl}")

print("\nSilver layer complete. Clean Delta tables are now queryable via Spark and the SQL endpoint.")
bids = (raw["sub_bids"]
    .withColumn("csi_division", clean_str("csi_division"))
    .withColumn("bid_amount",   num("bid_amount"))
    .withColumn("awarded",      to_bool("awarded"))
    .withColumn("bid_date",     parse_messy_date("bid_date"))
    .dropDuplicates(["bid_id"])
)

safety = (raw["safety_incidents"]
    .withColumn("incident_type",  clean_str("incident_type"))
    .withColumn("severity",       clean_str("severity"))
    .withColumn("trade_involved", clean_str("trade_involved"))
    .withColumn("root_cause",     clean_str("root_cause"))
    .withColumn("description",    clean_str("description"))
    .withColumn("incident_date",  parse_messy_date("incident_date"))
    .withColumn("lost_days",      num("lost_days", "int"))
    # binary target for the safety classifier
    .withColumn("is_lost_time",
                F.when(F.col("severity") == "Lost Time", True).otherwise(False))
    .dropDuplicates(["incident_id"])
)

for nm, df in [("schedule_tasks", sched), ("labor_timesheets", labor),
               ("sub_bids", bids), ("safety_incidents", safety)]:
    print(f"{nm:20s} {df.count():>7,} clean rows")

StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 9, Finished, Available, Finished, False)

wrote silver_projects
wrote silver_subcontractors
wrote silver_cost_line_items
wrote silver_schedule_tasks
wrote silver_labor_timesheets
wrote silver_sub_bids
wrote silver_safety_incidents

Silver layer complete. Clean Delta tables are now queryable via Spark and the SQL endpoint.
schedule_tasks         1,440 clean rows
labor_timesheets       3,411 clean rows
sub_bids               1,917 clean rows
safety_incidents         189 clean rows


#### Cell 8

In [8]:
# Validation summary (prove the cleanup worked)
print("=== SILVER DATA-QUALITY SUMMARY ===\n")

# date parsing: how many bad dates became null
bad_starts = projects_clean.filter(F.col("start_date").isNull()).count()
print(f"projects with unparseable start_date (quarantined as null): {bad_starts}")

# entity resolution result
print(f"vendors after resolution: {subs_clean.count()} "
      f"(from {raw['subcontractors'].count()} raw)")

# cost validity
invalid_costs = costs_clean.filter(~F.col("actual_valid")).count()
print(f"cost line items flagged invalid (null/negative/zero actual): {invalid_costs}")

# a clean analytical query now works end to end
print("\nMean overrun ratio by project type (valid rows only):")
(costs_clean.filter(F.col("actual_valid"))
    .join(projects_clean.select("project_id", "project_type"), "project_id")
    .groupBy("project_type")
    .agg(F.round(F.mean("overrun_ratio"), 3).alias("avg_overrun"),
         F.count("*").alias("n"))
    .orderBy(F.col("avg_overrun").desc())
    .show(truncate=False))


StatementMeta(, 5bf8199d-e3f3-449a-9730-6f8fa8ff7f93, 10, Finished, Available, Finished, False)

=== SILVER DATA-QUALITY SUMMARY ===

projects with unparseable start_date (quarantined as null): 0
vendors after resolution: 80 (from 92 raw)
cost line items flagged invalid (null/negative/zero actual): 50

Mean overrun ratio by project type (valid rows only):
+----------------------+-----------+---+
|project_type          |avg_overrun|n  |
+----------------------+-----------+---+
|Mission Critical      |1.262      |187|
|Data Center           |1.251      |85 |
|Healthcare            |1.218      |91 |
|Aviation              |1.209      |162|
|Industrial            |1.189      |143|
|Sports & Entertainment|1.187      |101|
|Higher Education      |1.148      |140|
|Commercial            |1.139      |108|
|Government/Civic      |1.138      |167|
|K-12 Education        |1.13       |114|
|NULL                  |1.111      |31 |
+----------------------+-----------+---+

